# LFM Semantic Segmentation Example Workflow
This notebook is an example workflow of doing semantic segmentation on visible light, UV, and static bands of Lunar data. 

You can get started with this notebook by downloading it with:

```bash
wget https://raw.githubusercontent.com/nasa-nccs-hpda/lfm/refs/heads/main/notebooks/semantic_seg.ipynb
```

The notebook requires some code from the [LFM repo](https://github.com/nasa-nccs-hpda/lfm/tree/main). To install this code, navigate to the top of the JupyterHub user interface, and click on the box with the + symbol. Then scroll down, and choose the "Terminal" (under Other section, first option). 

Then, if it's your first time using any of the LFM notebooks, run:

```bash
cd path/to/lfm/notebooks
git clone https://github.com/nasa-nccs-hpda/lfm.git
```

Or, if you've already run the above command, run:

```bash
cd path/to/lfm/notebooks
git pull https://github.com/nasa-nccs-hpda/lfm/tree/main
```

This will get you the most up-to-date code to support the notebook. 

**See the README in the [repo](https://github.com/nasa-nccs-hpda/lfm)** for more info on how to use this notebook, and more on the process of training the model. 

## Imports, Dino Repo Clone

In [ ]:
!rm -rf ~/.local/lib/python*

In [ ]:
!pip install rasterio  # transformers torchmetrics # For H100 environment

In [ ]:
import os
import sys
import torch
import subprocess
from glob import glob
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.image as mpimg

%matplotlib inline

In [ ]:
sys.path.insert(0, "/explore/nobackup/people/ajkerr1/Lunar_FM/latest_repo/lfm")

from lfm.tasks.sem_segmentation.sseg_model import DINOSegmentation, load_dinov3_encoder
from lfm.tasks.sem_segmentation.sseg_dataset import get_dataloaders
from lfm.tasks.sem_segmentation.sseg_driver import train_model

## Main workflow

1. Define user-configured variables
2. Create dataloaders from files on /explore/nobackup/.
3. Load DinoV3 encoder, create encoder/decoder finetuning model.
4. Train model, print training stats, and visualize results. 

### User Config

In [ ]:
# Local image of DinoV3 encoder
weights_local_checkpoint = '/explore/nobackup/projects/ilab/models/dinov3/dinov3_vitl16_pretrain_sat493m-eadcf0ff.pth'

# Data paths
INPUT_ROOT_DIR = "/explore/nobackup/projects/lfm/model_inputs/300_300_inputs/kaguya_static_all_wac/sem_seg"
IMAGE_DIR = f"{INPUT_ROOT_DIR}/chips"  # Where input images are stored
LABEL_DIR = f"{INPUT_ROOT_DIR}/labels"  # Where input labels are stored
IMAGE_FILE_TYPE = ".tif"
LABEL_FILE_TYPE = ".npy"

# Output dir (this will be created automatically)
OUTPUT_DIR = "./outputs/sem_seg_no_norm"  # Change this if you want a specific path
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

# Dataset parameters
MAX_SAMPLES = 500  # Set to None to use all samples, or an integer to limit
TRAIN_SPLIT = 0.8  # 80% train, 20% validation

# Training hyperparameters
BATCH_SIZE = 16  # Number of images fed into the model at a time
NUM_EPOCHS = 100  # 10 is the default, increase for more model learning
BASE_LR = 5e-5  # Starting learning rate
WEIGHT_DECAY = 1e-3  # Weight decay of optimizer
NUM_WORKERS = 8  # Used for parallelization
LOSS_TYPE = "focal_dice"  # Combined Dice + Binary CE loss
WARMUP_EPOCHS = 10  # Number of epochs for warmup LR scheduler
PATIENCE = 100  # How many epochs wait before early stopping
NORMALIZE_INPUTS = False  # Whether to use data mean/std to z-score norm data

# Model parameters
TARGET_SIZE = (304, 304)  # Input size for DINO model
FREEZE_ENCODER = False  # Whether to keep the DinoV3 weights frozen
NUM_BANDS = 12  # Number of bands to use on the model; (3, 5, 7, 12) all valid
BAND_FILTER = None  # Bands to keep, in order of filtering

# Visualization and saving
CHECKPOINT_EVERY = 10  # Save checkpoint every N epochs
VISUALIZE_EVERY = 10  # Create visualizations every N epochs

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

### Create dataloaders

In [ ]:
# ============================================================================
# CREATE DATALOADERS
# ============================================================================

print("\n" + "="*60)
print("STEP 1: Creating dataloaders.")
print("="*60)

train_loader, val_loader, MEAN, STD = get_dataloaders(
    image_dir=IMAGE_DIR,
    label_dir=LABEL_DIR,
    batch_size=BATCH_SIZE,
    train_split=TRAIN_SPLIT,
    num_workers=NUM_WORKERS,
    target_size=TARGET_SIZE,
    max_samples=MAX_SAMPLES,
    seed=42,
    stats_save_dir=OUTPUT_DIR,
    input_file_type=IMAGE_FILE_TYPE,
    label_file_type=LABEL_FILE_TYPE,
    debug=True,
    band_filter=BAND_FILTER,
    output_dir=OUTPUT_DIR,
    normalize_inputs=NORMALIZE_INPUTS
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

### Load Encoder and Create Model

In [ ]:
print("\n" + "="*60)
print("STEP 2: Loading DINO encoder and creating model.")
print("="*60)

encoder = load_dinov3_encoder(weights_local_checkpoint, device)

In [ ]:
# Create model with DINO segmentation head, UNet decoder (see model.py)
print("Creating DINO segmentation model with UNet decoder...")
model = DINOSegmentation(
    encoder=encoder,  # Use DINOv3 head
    num_classes=2,  # Binary segmentaton (crater, not crater)
    img_size=TARGET_SIZE,
    num_bands=NUM_BANDS,  # Use flexible embeddings for 5/7-band vis data
).to(device)

# Unfreeze encoder for full fine-tuning if desired
if FREEZE_ENCODER:
    for param in model.encoder.parameters():
        param.requires_grad = False
    print("Encoder frozen (only decoder will be trained).")
else:
    print("Encoder unfrozen! Full model will be trained.")

### Run Training

In [ ]:
print("\n" + "="*60)
print("Starting training.")
print("="*60)

train_losses, val_losses = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    mode="train",
    output_dir=OUTPUT_DIR,
    num_epochs=NUM_EPOCHS,
    learning_rate=BASE_LR,
    weight_decay=WEIGHT_DECAY,
    checkpoint_every=CHECKPOINT_EVERY,
    visualize_every=VISUALIZE_EVERY,
    loss_type=LOSS_TYPE,
    device=device,
    warmup_epochs=WARMUP_EPOCHS,
    early_stopping_patience=PATIENCE,
    max_grad_norm=1.0
)

## Display some of the output visualizations

The training of the model is already producing some visualizations every N epochs.
Here we open some of the visualizations to look at them from the notebook.

In [ ]:
visualization_dir = os.path.join(OUTPUT_DIR, 'visualizations')
visualization_filenames = sorted(glob(os.path.join(visualization_dir, '*.png')))

In [ ]:
for vis_filename in visualization_filenames:
    img = mpimg.imread(vis_filename)
    plt.figure(figsize=(16, 14))
    plt.imshow(img)
    plt.axis("off")
    plt.show()